# Remove Cache and Model Dataset
Ensures a fresh start

In [1]:
rm -rf ../../data/aemo_cache

SyntaxError: invalid syntax (11068920.py, line 1)

In [ ]:
rm -rf ../../data/sa1_model_dataset.csv

# NEM SA1 — Shared Dataset Build
**Purpose:**  
This notebook is the single source of truth for building the shared modelling dataset.

It:
1. Downloads the required AEMO dispatch tables
2. Filters to `SA1`
3. Removes intervention intervals
4. Merges all tables into one dataset
5. Engineers `CURTAILMENT_MW`
6. Resamples to 30-minute intervals
7. Saves the final dataset to `../../data/sa1_merged_eda.csv`

This output is used by downstream analysis and modelling notebooks.

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
from nemosis import dynamic_data_compiler

warnings.filterwarnings('ignore')

CACHE = '../../data/aemo_cache'
START = '2023/01/01 00:00:00'
END   = '2024/12/31 00:00:00'

os.makedirs('../../data', exist_ok=True)
os.makedirs(CACHE, exist_ok=True)

print('Setup complete.')
print(f'Data cache/output folder: {CACHE}')
print(f'Date range: {START} -> {END}')

Setup complete.
Data cache/output folder: ../../data/aemo_cache
Date range: 2023/01/01 00:00:00 -> 2024/12/31 00:00:00


## Extract source tables

Pull the three required AEMO tables:

- `DISPATCHPRICE`
- `DISPATCHREGIONSUM`
- `DISPATCHINTERCONNECTORRES`

Filter to SA1 and remove intervention intervals before merging.

In [3]:
# --- DISPATCHPRICE ---
# RRP is the target. FCAS prices (RAISE/LOWER) reflect system stress
# and may co-move with energy price extremes.
prices = dynamic_data_compiler(
    START, END, 'DISPATCHPRICE', CACHE,
    select_columns=[
        'SETTLEMENTDATE', 'REGIONID', 'INTERVENTION',
        'RRP',
        'RAISE6SECRRP', 'RAISE60SECRRP', 'RAISE5MINRRP', 'RAISEREGRRP',
        'LOWER6SECRRP', 'LOWER60SECRRP', 'LOWER5MINRRP', 'LOWERREGRRP',
    ],
    filter_cols=['REGIONID'],
    filter_values=(['SA1'],)
)
prices = prices[prices['INTERVENTION'] == 0].drop(columns=['REGIONID', 'INTERVENTION'])
print(f'Prices: {len(prices):,} rows')

INFO: Compiling data for table DISPATCHPRICE
INFO: Downloading data for table DISPATCHPRICE, year 2022, month 12
INFO: Creating feather file for DISPATCHPRICE, 2022, 12
INFO: Downloading data for table DISPATCHPRICE, year 2023, month 01
INFO: Creating feather file for DISPATCHPRICE, 2023, 01
INFO: Downloading data for table DISPATCHPRICE, year 2023, month 02
INFO: Creating feather file for DISPATCHPRICE, 2023, 02
INFO: Downloading data for table DISPATCHPRICE, year 2023, month 03
INFO: Creating feather file for DISPATCHPRICE, 2023, 03
INFO: Downloading data for table DISPATCHPRICE, year 2023, month 04
INFO: Creating feather file for DISPATCHPRICE, 2023, 04
INFO: Downloading data for table DISPATCHPRICE, year 2023, month 05
INFO: Creating feather file for DISPATCHPRICE, 2023, 05
INFO: Downloading data for table DISPATCHPRICE, year 2023, month 06
INFO: Creating feather file for DISPATCHPRICE, 2023, 06
INFO: Downloading data for table DISPATCHPRICE, year 2023, month 07
INFO: Creating feat

In [4]:
# --- DISPATCHREGIONSUM ---
# UIGF = unconstrained intermittent generation forecast (wind/solar potential)
# SEMISCHEDULE_CLEAREDMW = what was actually dispatched from semi-scheduled (wind/solar)
# SEMISCHEDULE_COMPLIANCEMW = semi-scheduled MW where cap is being enforced (curtailment signal)
# Gap between UIGF and SEMISCHEDULE_CLEAREDMW = curtailment pressure
# NETINTERCHANGE = net MW flowing out of SA (positive = exporting)
# EXCESSGENERATION = MW in excess of load + exports (most direct negative price signal)
regionsum = dynamic_data_compiler(
    START, END, 'DISPATCHREGIONSUM', CACHE,
    select_columns=[
        'SETTLEMENTDATE', 'REGIONID', 'INTERVENTION',
        'TOTALDEMAND',
        'AVAILABLEGENERATION',
        'AVAILABLELOAD',
        'DISPATCHABLEGENERATION',
        'DISPATCHABLELOAD',
        'NETINTERCHANGE',
        'EXCESSGENERATION',
        'INITIALSUPPLY',
        'CLEAREDSUPPLY',
        'TOTALINTERMITTENTGENERATION',
        'DEMANDFORECAST',
        'UIGF',
        'SEMISCHEDULE_CLEAREDMW',
        'SEMISCHEDULE_COMPLIANCEMW',
    ],
    filter_cols=['REGIONID'],
    filter_values=(['SA1'],)
)
regionsum = regionsum[regionsum['INTERVENTION'] == 0].drop(columns=['REGIONID', 'INTERVENTION'])
print(f'Regionsum: {len(regionsum):,} rows')

INFO: Compiling data for table DISPATCHREGIONSUM
INFO: Downloading data for table DISPATCHREGIONSUM, year 2022, month 12
INFO: Creating feather file for DISPATCHREGIONSUM, 2022, 12
INFO: Downloading data for table DISPATCHREGIONSUM, year 2023, month 01
INFO: Creating feather file for DISPATCHREGIONSUM, 2023, 01
INFO: Downloading data for table DISPATCHREGIONSUM, year 2023, month 02
INFO: Creating feather file for DISPATCHREGIONSUM, 2023, 02
INFO: Downloading data for table DISPATCHREGIONSUM, year 2023, month 03
INFO: Creating feather file for DISPATCHREGIONSUM, 2023, 03
INFO: Downloading data for table DISPATCHREGIONSUM, year 2023, month 04
INFO: Creating feather file for DISPATCHREGIONSUM, 2023, 04
INFO: Downloading data for table DISPATCHREGIONSUM, year 2023, month 05
INFO: Creating feather file for DISPATCHREGIONSUM, 2023, 05
INFO: Downloading data for table DISPATCHREGIONSUM, year 2023, month 06
INFO: Creating feather file for DISPATCHREGIONSUM, 2023, 06
INFO: Downloading data for 

In [5]:
# --- DISPATCHINTERCONNECTORRES ---
# SA1 has two interconnectors:
#   V-SA       = Heywood (SA <-> VIC), ~650 MW, primary route
#   V-SA-MNSP1 = Murraylink (SA <-> VIC via DC), ~220 MW
# When both hit export limits, SA is effectively islanded -> prices go negative
interconnectors = dynamic_data_compiler(
    START, END, 'DISPATCHINTERCONNECTORRES', CACHE,
    select_columns=['SETTLEMENTDATE', 'INTERCONNECTORID', 'INTERVENTION', 'MWFLOW', 'MWLOSSES'],
    filter_cols=['INTERCONNECTORID'],
    filter_values=(['V-SA', 'V-SA-MNSP1'],)
)
interconnectors = interconnectors[interconnectors['INTERVENTION'] == 0].drop(columns='INTERVENTION')

# Pivot to wide format — one row per SETTLEMENTDATE
interconnectors_wide = interconnectors.pivot_table(
    index='SETTLEMENTDATE',
    columns='INTERCONNECTORID',
    values=['MWFLOW', 'MWLOSSES']
).reset_index()
interconnectors_wide.columns = ['_'.join(col).strip('_') for col in interconnectors_wide.columns]
print(f'Interconnectors (wide): {len(interconnectors_wide):,} rows')
print(f'Columns: {interconnectors_wide.columns.tolist()}')

INFO: Compiling data for table DISPATCHINTERCONNECTORRES
INFO: Downloading data for table DISPATCHINTERCONNECTORRES, year 2022, month 12
INFO: Creating feather file for DISPATCHINTERCONNECTORRES, 2022, 12
INFO: Downloading data for table DISPATCHINTERCONNECTORRES, year 2023, month 01
INFO: Creating feather file for DISPATCHINTERCONNECTORRES, 2023, 01
INFO: Downloading data for table DISPATCHINTERCONNECTORRES, year 2023, month 02
INFO: Creating feather file for DISPATCHINTERCONNECTORRES, 2023, 02
INFO: Downloading data for table DISPATCHINTERCONNECTORRES, year 2023, month 03
INFO: Creating feather file for DISPATCHINTERCONNECTORRES, 2023, 03
INFO: Downloading data for table DISPATCHINTERCONNECTORRES, year 2023, month 04
INFO: Creating feather file for DISPATCHINTERCONNECTORRES, 2023, 04
INFO: Downloading data for table DISPATCHINTERCONNECTORRES, year 2023, month 05
INFO: Creating feather file for DISPATCHINTERCONNECTORRES, 2023, 05
INFO: Downloading data for table DISPATCHINTERCONNECTOR

In [ ]:
print(f"Prices date range: {prices.SETTLEMENTDATE.min()} to {prices.SETTLEMENTDATE.max()}")
print(f"Prices rows: {len(prices):,}")

## Merge, engineer, resample, and save

Use the price table as the anchor, then merge all other signals onto it.

In [6]:
df = prices.copy()
df = df.merge(regionsum, on='SETTLEMENTDATE', how='left')
df = df.merge(interconnectors_wide, on='SETTLEMENTDATE', how='left')

# Sort by time and set index
df = df.sort_values('SETTLEMENTDATE').reset_index(drop=True)
df['SETTLEMENTDATE'] = pd.to_datetime(df['SETTLEMENTDATE'])

# Derived feature: curtailment pressure (how much wind/solar is being held back)
# Positive value = generation is being curtailed
df['CURTAILMENT_MW'] = df['UIGF'] - df['SEMISCHEDULE_CLEAREDMW']

# Resample to 30-minute intervals for consistent downstream analysis
df = df.set_index('SETTLEMENTDATE').resample('30min').mean().reset_index()

# Save shared dataset
output_path = '../../data/sa1_model_dataset.csv'
df.to_csv(output_path, index=False)

print(f'Final dataset: {len(df):,} rows x {df.shape[1]} columns')
print(f'Date range: {df.SETTLEMENTDATE.min()} to {df.SETTLEMENTDATE.max()}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nSaved and ouputed FIle: {output_path}')

Final dataset: 35,041 rows x 27 columns
Date range: 2023-01-01 00:00:00 to 2024-12-31 00:00:00

Columns: ['SETTLEMENTDATE', 'RRP', 'RAISE6SECRRP', 'RAISE60SECRRP', 'RAISE5MINRRP', 'RAISEREGRRP', 'LOWER6SECRRP', 'LOWER60SECRRP', 'LOWER5MINRRP', 'LOWERREGRRP', 'TOTALDEMAND', 'AVAILABLEGENERATION', 'AVAILABLELOAD', 'DEMANDFORECAST', 'DISPATCHABLEGENERATION', 'DISPATCHABLELOAD', 'NETINTERCHANGE', 'EXCESSGENERATION', 'INITIALSUPPLY', 'CLEAREDSUPPLY', 'TOTALINTERMITTENTGENERATION', 'UIGF', 'SEMISCHEDULE_CLEAREDMW', 'SEMISCHEDULE_COMPLIANCEMW', 'MWFLOW_V-SA', 'MWLOSSES_V-SA', 'CURTAILMENT_MW']

Saved and ouputed FIle: ../../data/sa1_model_dataset.csv
